# 05_data_analysis

En este notebook se responden las preguntas de negocio usando la tabla `ANALYTICS.OBT_TRIPS` con Spark.

Las consultas se parametrizan con:
- años
- meses
- servicios

Además, la conexión usa variables de ambiente para Snowflake.

## Configuración y conexión

In [43]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession

In [44]:
YEARS = list(range(2015, 2026))
MONTHS = list(range(1, 13))
SERVICES = ["yellow", "green"]

DATABASE = os.getenv("SNOWFLAKE_DATABASE")
ANALYTICS_SCHEMA = os.getenv("SNOWFLAKE_SCHEMA_ANALYTICS")
OBT_TABLE = "OBT_TRIPS"

assert DATABASE is not None, "Falta SNOWFLAKE_DATABASE"
assert ANALYTICS_SCHEMA is not None, "Falta SNOWFLAKE_SCHEMA_ANALYTICS"

In [45]:
try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("P3 Data Analysis")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4",
            "net.snowflake:snowflake-jdbc:3.13.30"
        ])
    )
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.timestampType", "TIMESTAMP_LTZ")
    .getOrCreate()
)

spark

In [46]:
sfOptionsAnalytics = {
    "sfURL": f"{os.getenv('SNOWFLAKE_ACCOUNT')}.snowflakecomputing.com",
    "sfUser": os.getenv("SNOWFLAKE_USER"),
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD"),
    "sfDatabase": DATABASE,
    "sfSchema": ANALYTICS_SCHEMA,
    "sfWarehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "sfRole": os.getenv("SNOWFLAKE_ROLE"),
}

In [47]:
services_sql = ", ".join([f"'{s}'" for s in SERVICES])
years_sql = ", ".join([str(y) for y in YEARS])
months_sql = ", ".join([str(m) for m in MONTHS])

query = f"""
SELECT *
FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
WHERE source_year IN ({years_sql})
  AND source_month IN ({months_sql})
  AND service_type IN ({services_sql})
"""

obt = (
    spark.read
    .format("snowflake")
    .options(**sfOptionsAnalytics)
    .option("query", query)
    .load()
)

obt.createOrReplaceTempView("obt_trips")
print("Schema loaded:")
obt.printSchema()

Schema loaded:
root
 |-- PICKUP_DATETIME: timestamp (nullable = true)
 |-- DROPOFF_DATETIME: timestamp (nullable = true)
 |-- PICKUP_DATE: date (nullable = true)
 |-- PICKUP_HOUR: decimal(38,0) (nullable = true)
 |-- DROPOFF_DATE: date (nullable = true)
 |-- DROPOFF_HOUR: decimal(38,0) (nullable = true)
 |-- DAY_OF_WEEK: decimal(38,0) (nullable = true)
 |-- MONTH: decimal(38,0) (nullable = true)
 |-- YEAR: decimal(38,0) (nullable = true)
 |-- PU_LOCATION_ID: decimal(38,0) (nullable = true)
 |-- PU_ZONE: string (nullable = true)
 |-- PU_BOROUGH: string (nullable = true)
 |-- DO_LOCATION_ID: decimal(38,0) (nullable = true)
 |-- DO_ZONE: string (nullable = true)
 |-- DO_BOROUGH: string (nullable = true)
 |-- SERVICE_TYPE: string (nullable = true)
 |-- VENDOR_ID: decimal(38,0) (nullable = true)
 |-- VENDOR_NAME: string (nullable = true)
 |-- RATE_CODE_ID: decimal(38,0) (nullable = true)
 |-- RATE_CODE_DESC: string (nullable = true)
 |-- PAYMENT_TYPE: decimal(38,0) (nullable = true)
 |-- PA

## Pregunta a. Top 10 zonas de pickup por volumen mensual

In [48]:
q_a = spark.sql("""
WITH ranked AS (
    SELECT
        source_year,
        source_month,
        pu_zone,
        COUNT(*) AS total_trips,
        ROW_NUMBER() OVER (
            PARTITION BY source_year, source_month
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM obt_trips
    GROUP BY source_year, source_month, pu_zone
)
SELECT
    source_year,
    source_month,
    pu_zone,
    total_trips
FROM ranked
WHERE rn <= 10
ORDER BY source_year, source_month, total_trips DESC
""")

q_a.show(100, truncate=False)

+-----------+------------+----------------------------+-----------+
|source_year|source_month|pu_zone                     |total_trips|
+-----------+------------+----------------------------+-----------+
|2016       |5           |Union Sq                    |374459     |
|2016       |5           |Murray Hill                 |368889     |
|2016       |5           |East Village                |365151     |
|2016       |5           |Clinton East                |363809     |
|2016       |5           |Times Sq/Theatre District   |358687     |
|2016       |6           |Upper East Side South       |424655     |
|2016       |6           |Midtown Center              |394039     |
|2016       |6           |Upper East Side North       |385635     |
|2016       |6           |Penn Station/Madison Sq West|372514     |
|2016       |6           |Midtown East                |371094     |
|2016       |6           |Murray Hill                 |361830     |
|2016       |6           |Union Sq              

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta b. Top 10 zonas de dropoff por volumen mensual

In [49]:
q_b = spark.sql("""
WITH ranked AS (
    SELECT
        source_year,
        source_month,
        do_zone,
        COUNT(*) AS total_trips,
        ROW_NUMBER() OVER (
            PARTITION BY source_year, source_month
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM obt_trips
    GROUP BY source_year, source_month, do_zone
)
SELECT
    source_year,
    source_month,
    do_zone,
    total_trips
FROM ranked
WHERE rn <= 10
ORDER BY source_year, source_month, total_trips DESC
""")

q_b.show(100, truncate=False)

+-----------+------------+----------------------------+-----------+
|source_year|source_month|do_zone                     |total_trips|
+-----------+------------+----------------------------+-----------+
|2015       |1           |Midtown Center              |471817     |
|2015       |1           |Upper East Side North       |456225     |
|2015       |1           |Upper East Side South       |410200     |
|2015       |1           |Murray Hill                 |405982     |
|2015       |1           |Times Sq/Theatre District   |400836     |
|2015       |1           |Midtown East                |378792     |
|2015       |1           |Union Sq                    |369977     |
|2015       |1           |East Village                |369925     |
|2015       |1           |Clinton East                |355470     |
|2015       |1           |Penn Station/Madison Sq West|345424     |
|2015       |2           |Midtown Center              |461106     |
|2015       |2           |Upper East Side North 

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta c. Evolución mensual de total_amount y tip_pct por borough

In [50]:
q_c = spark.sql("""
SELECT
    source_year,
    source_month,
    pu_borough,
    SUM(total_amount) AS total_amount_sum,
    AVG(tip_pct) AS avg_tip_pct
FROM obt_trips
GROUP BY source_year, source_month, pu_borough
ORDER BY source_year, source_month, pu_borough
""")
q_c.show(100, truncate=False)

+-----------+------------+-------------+----------------+-------------+
|source_year|source_month|pu_borough   |total_amount_sum|avg_tip_pct  |
+-----------+------------+-------------+----------------+-------------+
|2016       |5           |Queens       |36798857.17     |13.506632397 |
|2016       |5           |Staten Island|22419.2         |23.849066959 |
|2016       |5           |Unknown      |2281341.55      |15.511081101 |
|2016       |5           |null         |233423.5        |469.931256927|
|2016       |6           |Bronx        |1119088.52      |4.541982665  |
|2016       |6           |Brooklyn     |12392737.91     |13.884329747 |
|2016       |6           |EWR          |20498.64        |141.20092511 |
|2016       |6           |Manhattan    |154307693.6     |15.026991833 |
|2016       |6           |Queens       |35046217.88     |16.476344702 |
|2016       |6           |Staten Island|25607.86        |15.005022727 |
|2016       |6           |Unknown      |2128963.68      |15.5033

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta d. Ticket promedio (avg total_amount) por service_type y mes

In [51]:
q_d = spark.sql("""
SELECT
    source_year,
    source_month,
    service_type,
    AVG(total_amount) AS avg_ticket
FROM obt_trips
GROUP BY source_year, source_month, service_type
ORDER BY source_year, source_month, service_type
""")
q_d.show(100, truncate=False)

+-----------+------------+------------+------------+
|source_year|source_month|service_type|avg_ticket  |
+-----------+------------+------------+------------+
|2016       |5           |yellow      |16.600902583|
|2016       |6           |green       |15.078434506|
|2016       |6           |yellow      |16.670237898|
|2016       |7           |green       |14.960827589|
|2016       |7           |yellow      |16.421501281|
|2016       |8           |green       |14.872124641|
|2016       |8           |yellow      |16.326383662|
|2016       |9           |green       |14.923286793|
|2016       |9           |yellow      |16.961342156|
|2016       |10          |green       |14.621574823|
|2016       |10          |yellow      |16.552242145|
|2016       |11          |green       |14.272024259|
|2016       |11          |yellow      |16.552577056|
|2016       |12          |green       |14.003471692|
|2016       |12          |yellow      |16.164914862|
|2017       |1           |green       |13.6513

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta e. Viajes por hora del día y día de semana (picos)

In [52]:
q_e = spark.sql("""
SELECT
    pickup_hour,
    day_of_week,
    COUNT(*) AS total_trips
FROM obt_trips
GROUP BY pickup_hour, day_of_week
ORDER BY day_of_week, pickup_hour
""")
q_e.show(200, truncate=False)

+-----------+-----------+-----------+
|pickup_hour|day_of_week|total_trips|
+-----------+-----------+-----------+
|21         |0          |4942485    |
|22         |0          |4373083    |
|23         |0          |3414758    |
|0          |1          |2264979    |
|1          |1          |1342564    |
|2          |1          |858001     |
|3          |1          |616929     |
|4          |1          |706510     |
|5          |1          |1144072    |
|6          |1          |2767877    |
|7          |1          |4704082    |
|8          |1          |5822931    |
|9          |1          |5668763    |
|10         |1          |5324106    |
|11         |1          |5388067    |
|12         |1          |5680168    |
|13         |1          |5778236    |
|14         |1          |6264905    |
|15         |1          |6416294    |
|16         |1          |6102411    |
|17         |1          |6905434    |
|9          |4          |6439931    |
|10         |4          |6047149    |
|11         

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta f. p50/p90 de trip_duration_min por borough de pickup

In [53]:
q_f = (
    spark.read
    .format("snowflake")
    .options(**sfOptionsAnalytics)
    .option(
        "query",
        f"""
        SELECT
            pu_borough,
            APPROX_PERCENTILE(trip_duration_min, 0.5) AS p50_trip_duration_min,
            APPROX_PERCENTILE(trip_duration_min, 0.9) AS p90_trip_duration_min
        FROM {OBT_TABLE}
        WHERE pu_borough IS NOT NULL
          AND trip_duration_min IS NOT NULL
          AND trip_duration_min BETWEEN 0 AND 300
        GROUP BY pu_borough
        ORDER BY pu_borough
        """
    )
    .load()
)

q_f.show(100, truncate=False)

+-------------+---------------------+---------------------+
|PU_BOROUGH   |P50_TRIP_DURATION_MIN|P90_TRIP_DURATION_MIN|
+-------------+---------------------+---------------------+
|Manhattan    |10.866323963         |25.041922472         |
|Brooklyn     |13.108416924         |32.598941435         |
|Queens       |25.077260358         |54.547113774         |
|Bronx        |13.974806485         |38.851154987         |
|Unknown      |11.236823348         |28.706032759         |
|Staten Island|27.3053481           |71.389189189         |
|EWR          |0.4996543779         |25.16648951          |
+-------------+---------------------+---------------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta g. avg_speed_mph por franja horaria (6–9, 17–20) y borough

In [54]:
q_g = spark.sql("""
SELECT
    CASE
        WHEN pickup_hour BETWEEN 6 AND 9 THEN '06-09'
        WHEN pickup_hour BETWEEN 17 AND 20 THEN '17-20'
        ELSE 'other'
    END AS franja_horaria,
    pu_borough,
    AVG(avg_speed_mph) AS avg_speed_mph
FROM obt_trips
WHERE pickup_hour BETWEEN 6 AND 9
   OR pickup_hour BETWEEN 17 AND 20
GROUP BY
    CASE
        WHEN pickup_hour BETWEEN 6 AND 9 THEN '06-09'
        WHEN pickup_hour BETWEEN 17 AND 20 THEN '17-20'
        ELSE 'other'
    END,
    pu_borough
ORDER BY franja_horaria, pu_borough
""")
q_g.show(100, truncate=False)

+--------------+-------------+-------------+
|franja_horaria|pu_borough   |avg_speed_mph|
+--------------+-------------+-------------+
|06-09         |EWR          |767.317419983|
|06-09         |Manhattan    |21.912337278 |
|17-20         |Queens       |56.253590997 |
|17-20         |Staten Island|129.23497239 |
|06-09         |Unknown      |38.555832045 |
|06-09         |null         |638.440384089|
|17-20         |EWR          |775.052715618|
|17-20         |Manhattan    |38.411875729 |
|06-09         |Queens       |363.334950407|
|06-09         |Staten Island|85.448174791 |
|17-20         |Bronx        |49.933896189 |
|17-20         |Brooklyn     |24.562971631 |
|17-20         |Unknown      |19.448946542 |
|17-20         |null         |601.628204827|
|06-09         |Bronx        |142.326211369|
|06-09         |Brooklyn     |53.761527639 |
+--------------+-------------+-------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta h. Participación por payment_type_desc y su relación con tip_pct

In [55]:
q_h = spark.sql("""
SELECT
    payment_type_desc,
    COUNT(*) AS total_trips,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_share,
    AVG(tip_pct) AS avg_tip_pct
FROM obt_trips
GROUP BY payment_type_desc
ORDER BY total_trips DESC
""")
q_h.show(100, truncate=False)

+-----------------+-----------+---------+---------------+
|payment_type_desc|total_trips|pct_share|avg_tip_pct    |
+-----------------+-----------+---------+---------------+
|Credit card      |577635060  |67.49    |23.497056661   |
|Unknown          |19411524   |2.27     |5.37966327     |
|No charge        |2931561    |0.34     |0.04425385866  |
|Dispute          |2100325    |0.25     |0.05700062699  |
|Cash             |253850000  |29.66    |0.0009491034928|
+-----------------+-----------+---------+---------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta i. ¿Qué rate_code_desc concentran mayor trip_distance y total_amount?

In [56]:
q_i = spark.sql("""
SELECT
    rate_code_desc,
    SUM(trip_distance) AS total_trip_distance,
    SUM(total_amount) AS total_amount_sum
FROM obt_trips
GROUP BY rate_code_desc
ORDER BY total_trip_distance DESC, total_amount_sum DESC
""")
q_i.show(100, truncate=False)

+------------------+-------------------+----------------+
|rate_code_desc    |total_trip_distance|total_amount_sum|
+------------------+-------------------+----------------+
|Standard rate     |3563199365.53      |13565111915.42  |
|JFK               |434564182.58       |1389298184.42   |
|Nassau/Westchester|21244571.35        |75272776.1      |
|Unknown           |831202391.57       |607840087.51    |
|Negotiated fare   |92038129.96        |197628541.78    |
|Newark            |28899233.43        |164089701.91    |
|Group ride        |8941.68            |75310.41        |
+------------------+-------------------+----------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta j. Mix yellow vs green por mes y borough

In [57]:
q_j = spark.sql("""
SELECT
    source_year,
    source_month,
    pu_borough,
    service_type,
    COUNT(*) AS total_trips,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (PARTITION BY source_year, source_month, pu_borough),
        2
    ) AS pct_mix
FROM obt_trips
GROUP BY source_year, source_month, pu_borough, service_type
ORDER BY CAST(source_year AS INT) ASC, CAST(source_month AS INT) ASC, pu_borough ASC, service_type ASC
""")
q_j.show(800, truncate=False)

+-----------+------------+-------------+------------+-----------+-------+
|source_year|source_month|pu_borough   |service_type|total_trips|pct_mix|
+-----------+------------+-------------+------------+-----------+-------+
|2015       |1           |Bronx        |green       |87343      |90.66  |
|2015       |1           |Bronx        |yellow      |9000       |9.34   |
|2015       |1           |Brooklyn     |green       |564018     |71.30  |
|2015       |1           |Brooklyn     |yellow      |226994     |28.70  |
|2015       |1           |EWR          |green       |13         |5.02   |
|2015       |1           |EWR          |yellow      |246        |94.98  |
|2015       |1           |Manhattan    |green       |426358     |3.55   |
|2015       |1           |Manhattan    |yellow      |11569729   |96.45  |
|2015       |1           |Queens       |green       |404153     |39.23  |
|2015       |1           |Queens       |yellow      |625971     |60.77  |
|2015       |1           |Staten Islan

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta k. Top 20 flujos PU→DO por volumen y su ticket promedio

In [58]:
q_k = spark.sql("""
SELECT
    pu_zone,
    do_zone,
    COUNT(*) AS total_trips,
    AVG(total_amount) AS avg_ticket
FROM obt_trips
GROUP BY pu_zone, do_zone
ORDER BY total_trips DESC
LIMIT 20
""")
q_k.show(20, truncate=False)

+-------------------------------+---------------------------------+-----------+-------------+
|pu_zone                        |do_zone                          |total_trips|avg_ticket   |
+-------------------------------+---------------------------------+-----------+-------------+
|Morrisania/Melrose             |Saint Michaels Cemetery/Woodside |6          |39.423333333 |
|Belmont                        |Willets Point                    |6          |49.581666667 |
|Marine Park/Floyd Bennett Field|Gowanus                          |6          |62.756666667 |
|Crotona Park                   |South Ozone Park                 |6          |55.575       |
|Bloomfield/Emerson Hill        |Midtown North                    |6          |63.183333333 |
|Van Cortlandt Park             |Windsor Terrace                  |6          |76.906666667 |
|East Flushing                  |Hudson Sq                        |6          |52.453333333 |
|Glen Oaks                      |Eltingville/Annadale/Prince

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta l. Distribución de passenger_count y efecto en total_amount

In [59]:
q_l = spark.sql("""
SELECT
    passenger_count,
    COUNT(*) AS total_trips,
    AVG(total_amount) AS avg_total_amount
FROM obt_trips
GROUP BY passenger_count
ORDER BY passenger_count
""")
q_l.show(100, truncate=False)

+---------------+-----------+----------------+
|passenger_count|total_trips|avg_total_amount|
+---------------+-----------+----------------+
|0              |25172040   |26.408510849    |
|1              |610374220  |18.278411042    |
|5              |33399309   |17.051852448    |
|2              |117991308  |19.67980702     |
|3              |32659478   |19.078616967    |
|4              |15791550   |20.015796706    |
|9              |994        |74.074818913    |
|8              |1870       |58.106748663    |
|6              |20535980   |16.891121684    |
|7              |1721       |50.627786171    |
+---------------+-----------+----------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta m. Impacto de tolls_amount y congestion_surcharge por zona

In [60]:
q_m = spark.sql("""
SELECT
    pu_zone,
    AVG(tolls_amount) AS avg_tolls_amount,
    AVG(congestion_surcharge) AS avg_congestion_surcharge,
    SUM(total_amount) AS total_amount_sum
FROM obt_trips
GROUP BY pu_zone
ORDER BY total_amount_sum DESC
""")
q_m.show(100, truncate=False)

+---------------------------------------------+----------------+------------------------+----------------+
|pu_zone                                      |avg_tolls_amount|avg_congestion_surcharge|total_amount_sum|
+---------------------------------------------+----------------+------------------------+----------------+
|UN/Turtle Bay South                          |0.3642870792    |2.448356267             |178278407.21    |
|null                                         |0.413678786     |1.607276302             |165165887.16    |
|Yorkville East                               |0.195151001     |2.460151743             |156029057.45    |
|Kips Bay                                     |0.1983573793    |2.439539299             |153594670.18    |
|Meatpacking/West Village West                |0.1556429445    |2.455335771             |148232777.07    |
|Little Italy/NoLiTa                          |0.05099161784   |2.462611135             |145159136.69    |
|Battery Park City                   

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta n. Proporción de viajes cortos vs largos por borough y estacionalidad

In [61]:
q_n = spark.sql("""
SELECT
    pu_borough,
    source_year,
    source_month,
    CASE
        WHEN trip_distance < 2 THEN 'short'
        WHEN trip_distance >= 2 AND trip_distance < 10 THEN 'medium'
        ELSE 'long'
    END AS trip_length_bucket,
    COUNT(*) AS total_trips
FROM obt_trips
GROUP BY
    pu_borough,
    source_year,
    source_month,
    CASE
        WHEN trip_distance < 2 THEN 'short'
        WHEN trip_distance >= 2 AND trip_distance < 10 THEN 'medium'
        ELSE 'long'
    END
ORDER BY pu_borough, source_year, source_month, trip_length_bucket
""")
q_n.show(200, truncate=False)

+----------+-----------+------------+------------------+-----------+
|pu_borough|source_year|source_month|trip_length_bucket|total_trips|
+----------+-----------+------------+------------------+-----------+
|Bronx     |2015       |1           |long              |3226       |
|Bronx     |2015       |1           |medium            |43107      |
|Bronx     |2015       |1           |short             |50010      |
|Bronx     |2015       |2           |long              |3860       |
|Bronx     |2015       |2           |medium            |57943      |
|Bronx     |2015       |2           |short             |67425      |
|Bronx     |2015       |3           |long              |4579       |
|Bronx     |2015       |3           |medium            |66913      |
|Bronx     |2015       |3           |short             |74027      |
|Bronx     |2015       |4           |long              |4672       |
|Bronx     |2015       |4           |medium            |58285      |
|Bronx     |2015       |4         

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta o. Diferencias por vendor en avg_speed_mph y trip_duration_min

In [62]:
q_o = spark.sql("""
SELECT
    vendor_name,
    AVG(avg_speed_mph) AS avg_speed_mph,
    AVG(trip_duration_min) AS avg_trip_duration_min
FROM obt_trips
GROUP BY vendor_name
ORDER BY avg_speed_mph DESC
""")
q_o.show(100, truncate=False)

+----------------------------+-------------+---------------------+
|vendor_name                 |avg_speed_mph|avg_trip_duration_min|
+----------------------------+-------------+---------------------+
|Helix                       |null         |0                    |
|Curb Mobility               |17.609661777 |18.695883339         |
|Unknown                     |10.73408348  |14.30829253          |
|Myle Technologies           |702.319509515|45.400860899         |
|Creative Mobile Technologies|69.63507524  |17.014308401         |
+----------------------------+-------------+---------------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta p. Relación método de pago ↔ tip_amount por hora

In [63]:
q_p = spark.sql("""
SELECT
    pickup_hour,
    payment_type_desc,
    AVG(tip_amount) AS avg_tip_amount,
    AVG(tip_pct) AS avg_tip_pct
FROM obt_trips
GROUP BY pickup_hour, payment_type_desc
ORDER BY pickup_hour, payment_type_desc
""")
q_p.show(200, truncate=False)

+-----------+-----------------+---------------+---------------+
|pickup_hour|payment_type_desc|avg_tip_amount |avg_tip_pct    |
+-----------+-----------------+---------------+---------------+
|12         |Cash             |0.0001219376421|0.0007977379871|
|12         |Credit card      |3.036153973    |23.056264799   |
|12         |Dispute          |0.01095373775  |0.06503403603  |
|12         |No charge        |0.004902936181 |0.05345337481  |
|12         |Unknown          |1.251263353    |6.262209774    |
|13         |Cash             |0.0001307406427|0.001002079052 |
|13         |Credit card      |3.138326889    |23.018245927   |
|13         |Dispute          |0.01040977611  |0.07405762855  |
|13         |No charge        |0.006655999147 |0.1656464999   |
|13         |Unknown          |1.282858821    |6.164963917    |
|14         |Cash             |0.0001336022917|0.0009392218692|
|14         |Credit card      |3.237799031    |22.854159894   |
|14         |Dispute          |0.0101071

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta q. Zonas con percentil 99 de duración/distancia fuera de rango (posible congestión/eventos)

In [64]:
q_q = (
    spark.read
    .format("snowflake")
    .options(**sfOptionsAnalytics)
    .option(
        "query",
        f"""
        SELECT
            pu_zone,
            APPROX_PERCENTILE(trip_duration_min, 0.99) AS p99_trip_duration_min,
            APPROX_PERCENTILE(trip_distance, 0.99) AS p99_trip_distance
        FROM {OBT_TABLE}
        WHERE pu_zone IS NOT NULL
          AND trip_duration_min IS NOT NULL
          AND trip_distance IS NOT NULL
          AND trip_duration_min BETWEEN 0 AND 300
          AND trip_distance BETWEEN 0 AND 100
        GROUP BY pu_zone
        ORDER BY p99_trip_duration_min DESC, p99_trip_distance DESC
        """
    )
    .load()
)

q_q.show(100, truncate=False)

+---------------------------------------------+---------------------+-----------------+
|PU_ZONE                                      |P99_TRIP_DURATION_MIN|P99_TRIP_DISTANCE|
+---------------------------------------------+---------------------+-----------------+
|JFK Airport                                  |95.182058038         |29.550847825     |
|South Beach/Dongan Hills                     |94.9856              |31.8344          |
|Allerton/Pelham Gardens                      |94.04068125          |25.756070248     |
|Glen Oaks                                    |93.949666667         |37.5604          |
|Brownsville                                  |93.902911184         |21.289653606     |
|Stapleton                                    |93.1853              |33.8306          |
|Country Club                                 |92.9567              |25.9369          |
|Howard Beach                                 |92.92468022          |33.676202381     |
|Whitestone                     

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta r. Yield por milla (total_amount/trip_distance) por borough y hora

In [65]:
q_r = spark.sql("""
SELECT
    pu_borough,
    pickup_hour,
    AVG(CASE WHEN trip_distance > 0 THEN total_amount / trip_distance END) AS avg_yield_per_mile
FROM obt_trips
GROUP BY pu_borough, pickup_hour
ORDER BY pu_borough, pickup_hour
""")
q_r.show(200, truncate=False)

+-------------+-----------+------------------+
|pu_borough   |pickup_hour|avg_yield_per_mile|
+-------------+-----------+------------------+
|Manhattan    |0          |9.536181485       |
|Manhattan    |1          |9.229442376       |
|Manhattan    |2          |8.973017387       |
|Manhattan    |3          |8.777521385       |
|Manhattan    |4          |8.555762526       |
|Manhattan    |5          |8.391666784       |
|Manhattan    |6          |8.314091204       |
|Manhattan    |7          |8.977853576       |
|Manhattan    |8          |10.048835795      |
|Manhattan    |9          |10.512486591      |
|Manhattan    |10         |10.629295332      |
|Manhattan    |11         |10.946020124      |
|Manhattan    |12         |11.056190703      |
|Manhattan    |13         |11.130761748      |
|Manhattan    |14         |11.067931253      |
|Manhattan    |15         |11.390855615      |
|Manhattan    |16         |14.316383555      |
|Manhattan    |17         |14.328566763      |
|Manhattan   

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta s. Cambios YoY en volumen y ticket promedio por service_type

In [66]:
q_s = spark.sql("""
WITH yearly AS (
    SELECT
        service_type,
        source_year,
        COUNT(*) AS total_trips,
        AVG(total_amount) AS avg_ticket
    FROM obt_trips
    GROUP BY service_type, source_year
)
SELECT
    service_type,
    source_year,
    total_trips,
    avg_ticket,
    LAG(total_trips) OVER (PARTITION BY service_type ORDER BY source_year) AS prev_year_trips,
    LAG(avg_ticket) OVER (PARTITION BY service_type ORDER BY source_year) AS prev_year_avg_ticket,
    ROUND(
        100.0 * (total_trips - LAG(total_trips) OVER (PARTITION BY service_type ORDER BY source_year))
        / NULLIF(LAG(total_trips) OVER (PARTITION BY service_type ORDER BY source_year), 0),
        2
    ) AS yoy_trips_pct,
    ROUND(
        100.0 * (avg_ticket - LAG(avg_ticket) OVER (PARTITION BY service_type ORDER BY source_year))
        / NULLIF(LAG(avg_ticket) OVER (PARTITION BY service_type ORDER BY source_year), 0),
        2
    ) AS yoy_avg_ticket_pct
FROM yearly
ORDER BY service_type, source_year
""")
q_s.show(100, truncate=False)

+------------+-----------+-----------+------------------+---------------+--------------------+-------------+------------------+
|service_type|source_year|total_trips|avg_ticket        |prev_year_trips|prev_year_avg_ticket|yoy_trips_pct|yoy_avg_ticket_pct|
+------------+-----------+-----------+------------------+---------------+--------------------+-------------+------------------+
|green       |2015       |18937057   |14.820283827629606|null           |null                |null         |null              |
|green       |2016       |16141551   |14.644182337868276|18937057       |14.820283827629606  |-14.76       |-1.19             |
|green       |2017       |11578265   |14.274481602381705|16141551       |14.644182337868276  |-28.27       |-2.52             |
|green       |2018       |8775698    |16.179843925805105|11578265       |14.274481602381705  |-24.21       |13.35             |
|green       |2019       |6128297    |18.311812027060697|8775698        |16.179843925805105  |-30.17    

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta t. Días con alta congestion_surcharge: efecto en total_amount vs días normales

In [67]:
q_t = spark.sql("""
WITH daily AS (
    SELECT
        pickup_date,
        AVG(congestion_surcharge) AS avg_congestion_surcharge,
        AVG(total_amount) AS avg_total_amount
    FROM obt_trips
    GROUP BY pickup_date
),
classified AS (
    SELECT
        pickup_date,
        avg_congestion_surcharge,
        avg_total_amount,
        CASE
            WHEN avg_congestion_surcharge >= percentile_approx(avg_congestion_surcharge, 0.9) OVER ()
            THEN 'high_congestion'
            ELSE 'normal'
        END AS day_type
    FROM daily
)
SELECT
    day_type,
    COUNT(*) AS total_days,
    AVG(avg_total_amount) AS avg_total_amount
FROM classified
GROUP BY day_type
ORDER BY day_type
""")
q_t.show(truncate=False)

+---------------+----------+------------------+
|day_type       |total_days|avg_total_amount  |
+---------------+----------+------------------+
|high_congestion|255       |24.053343348736387|
|normal         |3763      |20.636831714435807|
+---------------+----------+------------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Conclusión general

Resume aquí los hallazgos más importantes del análisis:
- patrones de demanda
- diferencias entre yellow y green
- zonas más activas
- comportamiento de propinas
- efectos de congestión, duración y velocidad

In [68]:
print("Notebook 05 listo")

Notebook 05 listo
